In [ ]:
import pandas as pd
import numpy as np
import sys, os
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

In [ ]:
df_train_raw = pd.read_csv('../data/raw/train.csv').drop('Id', axis=1)
df_train = df_train_raw.copy()
df_train = df_train.drop('SalePrice', axis=1)  # labels deleted 
print(f'start dimensions --> {df_train_raw.shape}')
df_train_raw.head()

In [ ]:
#Analisis de datos faltantes
missing_data = df_train_raw.isnull().sum()
missing_data = pd.DataFrame(missing_data[missing_data > 0], columns=['missing_count'])
missing_data['missing_percentage (%)'] = np.round(missing_data['missing_count'] / df_train_raw.shape[0] * 100, 2)
missing_data = missing_data.sort_values(by='missing_count', ascending=False)
print('fields with missing data:')
missing_data

In [ ]:
#Manejo de datos faltantes
#Paso inicial eliminar las columnas que tienen mas del 85% de datos faltantes

#LotFrontage: La longitud lineal (en pies) de la calle que bordea la propiedad.
#Alley: El tipo de acceso al callejón.
#MasVnrType: El tipo de revestimiento de mampostería.
#FireplaceQu: La calidad de la chimenea.
#PoolQC: La calidad de la piscina.
#Fence: La calidad de la valla.
#MiscFeature: Una característica miscelánea no cubierta en otras categorías.

print(f'start dimentions {df_train.shape}')
initial_columns = df_train.columns
df_train = df_train.dropna(thresh=len(df_train) * 0.85, axis=1)
columns_deleted = [col for col in initial_columns if col not in df_train.columns]
print(f'columns deleted ({len(columns_deleted)}) --> {columns_deleted}')

In [ ]:
#Columnas con datos faltantes resultantes de la eliminacion anterior
#Definir en una funcion
def missing_data_report(df):
    missing_res_deleted = df.isnull().sum()
    missing_res_deleted = missing_res_deleted[missing_res_deleted > 0].sort_values(ascending=True)
    return missing_res_deleted
res = missing_data_report(df_train)
res

In [ ]:
#imputacion simple
#Electrical: El sistema eléctrico. -> Valores faltantes imputados con la moda
df_train['Electrical'] = df_train['Electrical'].fillna(df_train['Electrical'].mode()[0])
#MasVnrArea: Área de revestimiento de mampostería en pies cuadrados. -> Valores faltantes imputados con (0)
df_train['MasVnrArea'] = df_train['MasVnrArea'].fillna(0)

In [ ]:
cond_not_basement = (df_train['TotalBsmtSF'] == 0) & (df_train['BsmtUnfSF'] == 0)
#BsmtCond: Evalúa la condición general del sótano. -> (NA) no tiene sótano
#BsmtQual: Evalúa la altura del sótano. -> (NA) no tiene sótano
##BsmtFinType1: Evaluación del tipo de acabado del sótano. -> (NA) no tiene sótano
#BsmtExposure: Refleja la cantidad de exposición al sótano al aire exterior. -> (NA) no tiene sótano
#BsmtFinType2: Evaluación del tipo de acabado del sótano (si hay dos tipos). -> (NA) no tiene sótano
cols_not_basement = ['BsmtCond', 'BsmtQual', 'BsmtFinType1', 'BsmtExposure', 'BsmtFinType2']
df_train.loc[cond_not_basement, cols_not_basement] = df_train.loc[cond_not_basement, cols_not_basement].fillna('NA')
fig, ax = plt.subplots(ncols=2, nrows=3)
fig.set_size_inches(15,8)

axisOne = sns.violinplot(x='BsmtCond', y='TotalBsmtSF', data=df_train, cut=0, ax=ax[0][0])
axisOne.set_title('Distribución de TotalBsmtSF por BsmtCond')
axisOne.set_xlabel('BsmtCond')
axisOne.set_ylabel('TotalBsmtSF')
axisOne.grid(axis='y', linestyle='--');

axisTwo = sns.violinplot(x='BsmtQual', y='TotalBsmtSF', data=df_train, cut=0, ax=ax[0][1])
axisTwo.set_title('Distribución de TotalBsmtSF por BsmtQual')
axisTwo.set_xlabel('BsmtQual')
axisTwo.set_ylabel('TotalBsmtSF')
axisTwo.grid(axis='y', linestyle='--');

axisThree = sns.violinplot(x='BsmtFinType1', y='TotalBsmtSF', data=df_train, cut=0, ax=ax[1][0])
axisThree.set_title('Distribución de TotalBsmtSF por BsmtFinType1')
axisThree.set_xlabel('BsmtFinType1')
axisThree.set_ylabel('TotalBsmtSF')
axisThree.grid(axis='y', linestyle='--');

axisFour = sns.violinplot(x='BsmtExposure', y='TotalBsmtSF', data=df_train, cut=0, ax=ax[1][1])
axisFour.set_title('Distribución de TotalBsmtSF por BsmtExposure')
axisFour.set_xlabel('BsmtExposure')
axisFour.set_ylabel('TotalBsmtSF')
axisFour.grid(axis='y', linestyle='--');
axisFive = sns.violinplot(x='BsmtFinType2', y='TotalBsmtSF', data=df_train, cut=0, ax=ax[2][0])
axisFive.set_title('Distribución de TotalBsmtSF por BsmtFinType2')
axisFive.set_xlabel('BsmtFinType2')
axisFive.set_ylabel('TotalBsmtSF')
axisFive.grid(axis='y', linestyle='--');
fig.delaxes(ax[2][1])
plt.tight_layout()

In [ ]:
#Columnas con datos faltantes resultantes de la eliminacion anterior

#TotalBsmtSF: Pies cuadrados totales del sótano.
#BsmtUnfSF: Pies cuadrados del sótano sin terminar.
#BsmtCond: Evalúa la condición general del sótano
#BsmtFinType1: Evaluación del tipo de acabado del sótano.
#BsmtFinSF1: Pies cuadrados del tipo de acabado 1 del sótano.
#BsmtFinType2: Evaluación del tipo de acabado del sótano (si hay dos tipos). (x)
#BsmtFinSF2: Pies cuadrados del tipo de acabado 2 del sótano.

#BsmtQual: Evalúa la altura del sótano.
#BsmtExposure: Refleja la cantidad de exposición al sótano al aire exterior. (x)
#BsmtFullBath: Número de baños completos en el sótano.
#BsmtHalfBath: Número de medios baños en el sótano.

df_train['BsmtExposure'] = df_train['BsmtExposure'].fillna(df_train.groupby(['BsmtCond', 'BsmtFinType1', 'BsmtFinType2', 'BsmtQual', 'BsmtFullBath', 'BsmtHalfBath'])['BsmtExposure']
                                                    .transform(lambda x: x.mode()[0] if not x.mode().empty else 'No'))
df_train['BsmtFinType2'] = df_train['BsmtFinType2'].fillna(df_train.groupby(['BsmtCond', 'BsmtQual', 'BsmtFullBath', 'BsmtHalfBath', 'BsmtExposure'])['BsmtFinType2']
                                                    .transform(lambda x: x.mode()[0] if not x.mode().empty else 'Unf'))

In [ ]:
#Imputacion de datos faltantes en columnas relacionadas con el garage ['GarageType', 'GarageYrBlt', 'GarageFinish', 'GarageQual', 'GarageCond'] -> ['GarageArea', 'GarageCars']
#GarageType ->  Ubicación del garaje. (NA) no tiene garage
#GarageYrBlt -> Año de construcción del garaje. (0) no tiene garage
#GarageFinish -> Acabado interior del garaje. (NA) no tiene garage
#GarageQual -> Calidad del garaje. (NA) no tiene garage
#GarageCond -> Condición del garaje. (NA) no tiene garage
list_categorical_not_garage = ['GarageType', 'GarageFinish', 'GarageQual', 'GarageCond']
list_numerical_not_garage = ['GarageYrBlt']
cond_not_garage = df_train['GarageArea'] == 0
df_train.loc[cond_not_garage, list_categorical_not_garage] = df_train.loc[cond_not_garage, list_categorical_not_garage].fillna('NA')
df_train.loc[cond_not_garage, list_numerical_not_garage] = df_train.loc[cond_not_garage, list_numerical_not_garage].fillna(0)

In [ ]:
fig, ax = plt.subplots(ncols=2, nrows=2)
fig.set_size_inches(15,8)
axisType = sns.violinplot(x='GarageType', y='GarageArea', data=df_train, cut=0, ax=ax[0][0])
axisType.set_title('Distribución del área del garaje por tipo de garaje')
axisType.set_xlabel('GarageType')
axisType.set_ylabel('GarageArea')
axisType.grid(axis='y', linestyle='--');
axisFinish = sns.violinplot(x='GarageFinish', y='GarageArea', data=df_train, cut=0, ax=ax[0][1])
axisFinish.set_title('Distribución del área del garaje por acabado del garaje')
axisFinish.set_xlabel('GarageFinish')
axisFinish.set_ylabel('GarageArea')
axisFinish.grid(axis='y', linestyle='--');
axisQual = sns.violinplot(x='GarageQual', y='GarageArea', data=df_train, cut=0, ax=ax[1][0])
axisQual.set_title('Distribución del área del garaje por calidad del garaje')
axisQual.set_xlabel('GarageQual')
axisQual.set_ylabel('GarageArea')
axisQual.grid(axis='y', linestyle='--');
axisCond = sns.violinplot(x='GarageCond', y='GarageArea', data=df_train, cut=0, ax=ax[1][1])
axisCond.set_title('Distribución del área del garaje por condición del garaje')
axisCond.set_xlabel('GarageCond')
axisCond.set_ylabel('GarageArea')
axisCond.grid(axis='y', linestyle='--');
plt.tight_layout()


In [ ]:
#Feature engineering
print(f'Numero de features --> {len(df_train.columns)}')
numeric_features = df_train.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = df_train.select_dtypes(include=[object]).columns.tolist()
print(f'numericas {len(numeric_features)} --> {numeric_features}')
print(f'categoricas {len(categorical_features)} --> {categorical_features}')

In [157]:
#Feature engineering (Categorical)

#MSZoning: Clasificación general de la zona de uso del suelo.
print('MSZoning: ')
#Considerar valores categoricos de MSZoning que no tienen ejemplos en el set de entrenamiento
one_hot_mszoning = pd.get_dummies(df_train['MSZoning'], prefix='MSZoning', dtype=int)
mszoning_cat_all = ['A', 'C (all)', 'FV', 'I', 'RH', 'RL', 'RP', 'RM']
mszoning_cat_train = df_train['MSZoning'].unique().tolist()
missing_cats = list(set(mszoning_cat_all) - set(mszoning_cat_train))
print(f'{"🟢" if len(missing_cats) == 0 else "🔴"} missing categories {missing_cats}')
print(f'🟢 {one_hot_mszoning.columns.to_list()}', end='\n\n')

""" Actual codificacion one hot encoding
Podriamos usar:
-> Ordinal Encoding
-> Codificación con Mapeo Manual (Manual Mapping)

Analisis de la variable:
1. Podriamos agrupar por categorias y ver que relevancia tienen en el precio de la vivienda
"""

#Street: Tipo de acceso a la propiedad
print('Street: ')
one_hot_street = pd.get_dummies(df_train['Street'], prefix='Street', dtype=int)
street_cat_all = ['Grvl', 'Pave']
street_cat_train = df_train['Street'].unique().tolist()
missing_cats = list(set(street_cat_all) - set(street_cat_train))
print(f'{"🟢" if len(missing_cats) == 0 else "🔴"} missing categories {missing_cats}')
print(f'🟢 {one_hot_street.columns.to_list()}', end='\n\n')

""" Actual codificacion one hot encoding
Podriamos usar:
-> Ordinal Encoding
-> Codificación con Mapeo Manual (Manual Mapping)

Analisis de la variable:
1. Podriamos agrupar por categorias y ver que relevancia tienen en el precio de la vivienda
2. Comparar los precios y verificar la distancia numerica en precios entre las categorias
"""

#LotShape: Forma general de la propiedad
print('LotShape: ')
one_hot_lotshape = pd.get_dummies(df_train['LotShape'], prefix='LotShape', dtype=int)
lot_shape_cat_all = ['Reg', 'IR1', 'IR2', 'IR3']
lot_shape_cat_train = df_train['LotShape'].unique().tolist()
missing_cats = list(set(lot_shape_cat_all) - set(lot_shape_cat_train))
print(f'{"🟢" if len(missing_cats) == 0 else "🔴"} missing categories {missing_cats}')
print(f'🟢 {one_hot_lotshape.columns.to_list()}', end='\n\n')
""" Actual codificacion one hot encoding
Podriamos usar:
-> Ordinal Encoding
-> Codificación con Mapeo Manual (Manual Mapping)

Analisis de la variable:
1. Podriamos agrupar por categorias y ver que relevancia tienen en el precio de la vivienda
2. Comparar los precios y verificar la distancia numerica en precios entre las categorias
"""

print('LandContour: ')
one_hot_landcontour = pd.get_dummies(df_train['LandContour'], prefix='LandContour', dtype=int)
land_countour_cat_all = ['Lvl', 'Bnk', 'HLS', 'Low']
land_countour_cat_train = df_train['LandContour'].unique().tolist()
missing_cats = list(set(land_countour_cat_all) - set(land_countour_cat_train))
print(f'{"🟢" if len(missing_cats) == 0 else "🔴"} missing categories {missing_cats}')
print(f'🟢 {one_hot_landcontour.columns.to_list()}', end='\n\n')
""" Actual codificacion one hot encoding
Podriamos usar:
-> Ordinal Encoding
-> Codificación con Mapeo Manual (Manual Mapping)

Analisis de la variable:
1. Podriamos agrupar por categorias y ver que relevancia tienen en el precio de la vivienda
2. Comparar los precios y verificar la distancia numerica en precios entre las categorias
"""

print('Utilities: ')
one_hot_utilities = pd.get_dummies(df_train['Utilities'], prefix='Utilities', dtype=int)
utilities_cat_all = ['AllPub', 'NoSewr', 'NoSeWa', 'ELO']
utilities_cat_train = df_train['Utilities'].unique().tolist()
missing_cats = list(set(utilities_cat_all) - set(utilities_cat_train))
print(f'{"🟢" if len(missing_cats) == 0 else "🔴"} missing categories {missing_cats}')
print(f'🟢 {one_hot_utilities.columns.to_list()}', end='\n\n')
""" Actual codificacion one hot encoding
Podriamos usar:
-> Ordinal Encoding
-> Codificación con Mapeo Manual (Manual Mapping)

Analisis de la variable:
1. Podriamos agrupar por categorias y ver que relevancia tienen en el precio de la vivienda
2. Comparar los precios y verificar la distancia numerica en precios entre las categorias
"""

print('LotConfig: ')
one_hot_lotconfig = pd.get_dummies(df_train['LotConfig'], prefix='LotConfig', dtype=int)
lot_config_cat_all = ['Inside', 'Corner', 'CulDSac', 'FR2', 'FR3']
lot_config_cat_train = df_train['LotConfig'].unique().tolist()
missing_cats = list(set(lot_config_cat_all) - set(lot_config_cat_train))
print(f'{"🟢" if len(missing_cats) == 0 else "🔴"} missing categories {missing_cats}')
print(f'🟢 {one_hot_lotconfig.columns.to_list()}', end='\n\n')
""" Actual codificacion one hot encoding
Podriamos usar:
-> Ordinal Encoding
-> Codificación con Mapeo Manual (Manual Mapping)

Analisis de la variable:
1. Podriamos agrupar por categorias y ver que relevancia tienen en el precio de la vivienda
2. Comparar los precios y verificar la distancia numerica en precios entre las categorias
"""

print('LandSlope: ')
one_hot_landslope = pd.get_dummies(df_train['LandSlope'], prefix='LandSlope', dtype=int)
land_slope_cat_all = ['Gtl', 'Mod', 'Sev']
land_slope_cat_train = df_train['LandSlope'].unique().tolist()
missing_cats = list(set(land_slope_cat_all) - set(land_slope_cat_train))
print(f'{"🟢" if len(missing_cats) == 0 else "🔴"} missing categories {missing_cats}')
print(f'🟢 {one_hot_landslope.columns.to_list()}', end='\n\n')
""" Actual codificacion one hot encoding
Podriamos usar:
-> Ordinal Encoding
-> Codificación con Mapeo Manual (Manual Mapping)

Analisis de la variable:
1. Podriamos agrupar por categorias y ver que relevancia tienen en el precio de la vivienda
2. Comparar los precios y verificar la distancia numerica en precios entre las categorias
"""

print('end!')

MSZoning: 
🔴 missing categories ['A', 'I', 'RP']
🟢 ['MSZoning_C (all)', 'MSZoning_FV', 'MSZoning_RH', 'MSZoning_RL', 'MSZoning_RM']

Street: 
🟢 missing categories []
🟢 ['Street_Grvl', 'Street_Pave']

LotShape: 
🟢 missing categories []
🟢 ['LotShape_IR1', 'LotShape_IR2', 'LotShape_IR3', 'LotShape_Reg']

LandContour: 
🟢 missing categories []
🟢 ['LandContour_Bnk', 'LandContour_HLS', 'LandContour_Low', 'LandContour_Lvl']

Utilities: 
🔴 missing categories ['NoSewr', 'ELO']
🟢 ['Utilities_AllPub', 'Utilities_NoSeWa']

LotConfig: 
🟢 missing categories []
🟢 ['LotConfig_Corner', 'LotConfig_CulDSac', 'LotConfig_FR2', 'LotConfig_FR3', 'LotConfig_Inside']

LandSlope: 
🟢 missing categories []
🟢 ['LandSlope_Gtl', 'LandSlope_Mod', 'LandSlope_Sev']

end!
